# Решения: Проект 21 — глобальное сравнение (USD / PPP / грейды)

Решения упражнений из `21_Global_IT_Comparison.ipynb`. Базовый набор данных
и модель воспроизведены ниже.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

ROWS = [
    ('США','Вашингтон (WA)','USD',158000,0.25,1650,108),
    ('США','Техас (TX)','USD',130000,0.24,1350,93),
    ('США','Калифорния (CA)','USD',159000,0.34,1900,135),
    ('США','Нью-Йорк (NY)','USD',137000,0.31,1550,125),
    ('Россия','Москва','RUB',3000000,0.13,60000,45),
    ('Россия','Новосибирск','RUB',2160000,0.13,28000,33),
    ('Россия','Казань','RUB',2100000,0.13,30000,33),
    ('ЕС','Люксембург','EUR',85000,0.29,1800,110),
    ('ЕС','Германия','EUR',68000,0.39,1250,100),
    ('ЕС','Нидерланды','EUR',66000,0.32,1900,108),
    ('ЕС','Ирландия','EUR',70000,0.28,2100,125),
    ('ЕС','Эстония','EUR',42000,0.21,800,75),
    ('ЕС','Польша','EUR',40000,0.24,950,58),
    ('ЕС','Португалия','EUR',35000,0.29,1300,70),
]
df = pd.DataFrame(ROWS, columns=['region','location','currency','gross_local','ded_rate','rent_local','PLI'])
FX = {'USD':1.0,'EUR':1.08,'RUB':1/92.0}
GRADES = {'junior':0.60,'middle':1.00,'senior':1.60}

def compute(df, fx, mult=1.0):
    d = df.copy()
    d['fx'] = d['currency'].map(fx)
    d['gross_usd'] = d['gross_local'] * d['fx'] * mult
    d['net_usd'] = d['gross_usd'] * (1 - d['ded_rate'])
    d['rent_usd_year'] = d['rent_local'] * 12 * d['fx']
    d['dispo_usd'] = d['net_usd'] - d['rent_usd_year']
    d['dispo_ppp'] = d['dispo_usd'] / (d['PLI'] / 100)
    return d

base = compute(df, FX)
print('Базовый рейтинг (middle, PPP):')
print(base.sort_values('dispo_ppp', ascending=False)[['location','dispo_usd','dispo_ppp']].head())

## Упражнение 1: Прогрессивный налог по грейдам

Заменим фиксированную `ded_rate` на прогрессивную шкалу для США (федеральная 2024 +
FICA 7.65%) и России (НДФЛ 2025). Для ЕС оставим фиксированную. Пересчитаем **senior**.

In [ ]:
FED = [(0,11600,0.10),(11600,47150,0.12),(47150,100525,0.22),(100525,191950,0.24),
       (191950,243725,0.32),(243725,609350,0.35),(609350,np.inf,0.37)]
def us_ded_rate(gross_usd):
    taxable = max(0.0, gross_usd - 14600); tax = 0.0
    for lo, hi, r in FED:
        if taxable > lo: tax += (min(taxable, hi) - lo) * r
        else: break
    return tax / gross_usd + 0.0765   # + FICA

NDFL = [(0,2_400_000,0.13),(2_400_000,5_000_000,0.15),(5_000_000,20_000_000,0.18),
        (20_000_000,50_000_000,0.20),(50_000_000,np.inf,0.22)]
def ru_ded_rate(gross_rub):
    tax = 0.0
    for lo, hi, r in NDFL:
        if gross_rub > lo: tax += (min(gross_rub, hi) - lo) * r
        else: break
    return tax / gross_rub

def compute_progressive(df, fx, mult):
    d = df.copy()
    d['fx'] = d['currency'].map(fx)
    d['gross_usd'] = d['gross_local'] * d['fx'] * mult
    rates = []
    for _, r in d.iterrows():
        if r['region'] == 'США': rates.append(us_ded_rate(r['gross_usd']))
        elif r['region'] == 'Россия': rates.append(ru_ded_rate(r['gross_local'] * mult))
        else: rates.append(r['ded_rate'])
    d['ded_rate'] = rates
    d['net_usd'] = d['gross_usd'] * (1 - d['ded_rate'])
    d['rent_usd_year'] = d['rent_local'] * 12 * d['fx']
    d['dispo_usd'] = d['net_usd'] - d['rent_usd_year']
    d['dispo_ppp'] = d['dispo_usd'] / (d['PLI'] / 100)
    return d

sen = compute_progressive(df, FX, GRADES['senior'])
top = sen.sort_values('dispo_ppp', ascending=False)[['location','ded_rate','dispo_ppp']].head(8).copy()
top['ded_rate'] = (top['ded_rate']*100).map('{:.1f}%'.format)
top['dispo_ppp'] = top['dispo_ppp'].map('${:,.0f}'.format)
print('SENIOR с прогрессивным налогом (топ-8, PPP):')
print(top.to_string(index=False))
print('\nУ senior в США эфф. ставка растёт (прогрессия), но высокий номинал сохраняет лидерство.')

## Упражнение 2: Цель — накопить в валюте (номинал USD, без PPP)

In [ ]:
nom = base.sort_values('dispo_usd', ascending=False)[['location','region','dispo_usd']].head(8).copy()
nom['3 года'] = (nom['dispo_usd'] * 3).map('${:,.0f}'.format)
nom['dispo_usd'] = nom['dispo_usd'].map('${:,.0f}'.format)
print('Топ-8 по НОМИНАЛЬНОМУ располагаемому доходу (для накоплений в USD):')
print(nom.to_string(index=False))
print('\nБез PPP лидируют США: максимум абсолютных долларов для накоплений/инвестиций.')

## Упражнение 3: Чувствительность к курсу валют

In [ ]:
print('Ранг локаций (PPP, middle) при разных курсах:')
scenarios = {'RUB сильный (70)':{'USD':1,'EUR':1.08,'RUB':1/70},
             'База (92)':FX,
             'RUB слабый (110)':{'USD':1,'EUR':1.08,'RUB':1/110},
             'EUR сильный (1.20)':{'USD':1,'EUR':1.20,'RUB':1/92}}
res = pd.DataFrame({'location': df['location'], 'region': df['region']})
for name, fx in scenarios.items():
    d = compute(df, fx)
    order = {loc:i+1 for i,loc in enumerate(d.sort_values('dispo_ppp',ascending=False)['location'])}
    res[name] = res['location'].map(order)
print(res.to_string(index=False))
print('\nСША устойчиво лидируют; позиции РФ и ЕС смещаются при изменении курсов.')

## Упражнение 4: Стоимость релокации и срок окупаемости

Сравним с базой (например, Москва): за сколько лет прирост располагаемого дохода
(PPP) окупит единовременные затраты на переезд.

In [ ]:
RELOCATION_COST = 15000  # USD, единовременно
home = base[base.location == 'Москва'].iloc[0]['dispo_ppp']
d = base.copy()
d['gain'] = d['dispo_ppp'] - home
d['payback_years'] = np.where(d['gain'] > 0, RELOCATION_COST / d['gain'], np.inf)
out = d[d.gain > 0].sort_values('payback_years')[['location','dispo_ppp','gain','payback_years']].head(8).copy()
out['dispo_ppp'] = out['dispo_ppp'].map('${:,.0f}'.format)
out['gain'] = out['gain'].map('${:,.0f}'.format)
out['payback_years'] = out['payback_years'].map('{:.2f} лет'.format)
print(f'Окупаемость релокации из Москвы (затраты ${RELOCATION_COST:,}):')
print(out.to_string(index=False))

## Упражнение 5: Персональный индекс (свой грейд и приоритеты)

In [ ]:
MY_GRADE = 'senior'
W = {'dispo_ppp':0.6, 'net_usd':0.4}   # важны и реальный комфорт, и абсолютный доход
d = compute(df, FX, GRADES[MY_GRADE])
def z(s): return (s - s.mean())/s.std(ddof=0)
d['idx'] = W['dispo_ppp']*z(d['dispo_ppp']) + W['net_usd']*z(d['net_usd'])
top5 = d.sort_values('idx', ascending=False)[['location','region','net_usd','dispo_ppp']].head(5).copy()
for col in ['net_usd','dispo_ppp']: top5[col] = top5[col].map('${:,.0f}'.format)
print(f'Мой персональный топ-5 (грейд={MY_GRADE}):')
print(top5.to_string(index=False))